# CSE5280 — Robot Arm Crowd Interference Simulation
**Author:** Michael Caruso  
**Course:** CSE5280 — Computer Vision & Simulation  
**Date:** Spring 2026

---

## Table of Contents
1. [Environment Setup](#setup)
2. [Mathematical Framework](#math)
3. [Building Geometry & Evacuation Model](#geometry)
4. [Cost Functions & Agent Dynamics](#cost)
5. [Robot Arm — Inverse Kinematics](#robot)
6. [Cluster Detection & Motion Prediction](#cluster)
7. [Simulation Loop with Robot Interference](#sim)
8. [Experiments & Analysis](#experiments)
9. [Discussion](#discussion)

---

### Overview

This notebook extends the building evacuation simulation by introducing a **robotic agent** that actively interferes with the crowd's escape.  
The robot is modeled as a 3-DOF arm whose **end-effector acts as a dynamic obstacle**.

At each timestep the robot:
1. Observes the particle distribution
2. Identifies particles near exits using **k-means clustering**
3. Predicts the cluster's future position via **velocity extrapolation**
4. Moves its end-effector toward the predicted intercept point using **Jacobian-based Inverse Kinematics**

Particles respond to the end-effector as a moving obstacle through an additional **Gaussian repulsion** cost term, producing emergent crowd–robot interaction.


In [ ]:
#@title # 1. Environment Setup { display-mode: "form" }
#@markdown Install vedo, download robot STL parts, set up virtual display.

!pip install -q vedo
!apt-get install -q -y xvfb freeglut3-dev ffmpeg > /dev/null 2>&1

import os, glob, sys
os.system("Xvfb :99 -screen 0 1920x1080x24 &")
os.environ["DISPLAY"] = ":99"

# Download robot STL parts
if not os.path.exists("robot/"):
    !wget -q -O robot.zip "https://www.dropbox.com/scl/fi/uewvrcempf2wf2jp7bcb8/robot.zip?rlkey=7uwz1ne94hxyinub8x16y93em&dl=1"
    !unzip -q robot.zip
    !rm robot.zip
    print("Robot parts downloaded.")
else:
    print("Robot parts already present.")

import vedo
from vedo import (load, Axes, Circle, Arrow, Sphere, Cylinder, Disc,
                  Box, Mesh, Plotter, LinearTransform, settings as vedo_settings)
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display as ipy_display
from sklearn.cluster import KMeans

vedo_settings.default_backend = "vtk"
print("Environment ready. vedo:", vedo.__version__)


## 2. Mathematical Framework <a name="math"></a>

### 2.1 Total Agent Cost

$$C_{\text{total}}(\mathbf{p}) = C_{\text{goal}} + C_{\text{walls}} + C_{\text{height}} + C_{\text{smooth}} + C_{\text{repulsion}} + C_{\text{robot}}$$

The new term $C_{\text{robot}}$ treats the end-effector as a dynamic obstacle:

$$C_{\text{robot}}(\mathbf{p}) = w_r \exp\!\left(-\frac{\|\mathbf{p} - \mathbf{e}(t)\|^2}{2\sigma_r^2}\right)$$

where $\mathbf{e}(t)$ is the end-effector position at time $t$.

---

### 2.2 Soft-Min Exit Selection

$$C_{\text{goal}}(\mathbf{p}) = -\tau \log\!\left(\sum_{i=1}^{2} \exp\!\left(-\frac{\|\mathbf{p} - \mathbf{p}_i^{\text{exit}}\|^2}{\tau}\right)\right)$$

---

### 2.3 K-Means Cluster Detection

At each step, particles within distance $d_{\text{exit}}$ of any exit are collected. K-means with $k=2$ is applied to find the dominant evacuation cluster:

$$\underset{\{\mu_k\}}{\arg\min} \sum_{k}\sum_{\mathbf{p} \in S_k} \|\mathbf{p} - \mu_k\|^2$$

---

### 2.4 Cluster Velocity Prediction

The cluster centroid velocity is estimated via finite difference:

$$\hat{\mathbf{v}}_k = \frac{\boldsymbol{\mu}_k^{(t)} - \boldsymbol{\mu}_k^{(t-1)}}{\Delta t}$$

The predicted intercept position at horizon $T_h$ steps ahead:

$$\hat{\mathbf{p}}_{\text{target}} = \boldsymbol{\mu}_k^{(t)} + T_h \cdot \hat{\mathbf{v}}_k$$

---

### 2.5 Jacobian Inverse Kinematics

The robot minimises $\|\mathbf{e}(\boldsymbol{\phi}) - \hat{\mathbf{p}}_{\text{target}}\|$ via iterative Newton updates:

$$\boldsymbol{\phi}^{k+1} = \boldsymbol{\phi}^k + J^+(\boldsymbol{\phi}^k)\,\lambda(\hat{\mathbf{p}}_{\text{target}} - \mathbf{e}^k)$$

where $J^+ = J^T(JJ^T + \lambda I)^{-1}$ is the damped pseudoinverse.


In [ ]:
#@title # 3. Building Geometry & Evacuation Model { display-mode: "form" }
#@markdown Defines the 2D floor plan used for the crowd simulation.
#@markdown The robot operates in this same coordinate space.

# ─── Building parameters (2D crowd sim uses XY plane) ─────────────────────────
FLOOR_W = 300.0   # Half-width  (matches robot coordinate scale ~550 units circle)
FLOOR_D = 250.0   # Half-depth
WALL_H  = 80.0    # Visual wall height for rendering

# Exits on the south face (y = -FLOOR_D)
EXIT_A = np.array([ 150.0, -FLOOR_D, 0.0])
EXIT_B = np.array([-150.0, -FLOOR_D, 0.0])
EXITS  = [EXIT_A, EXIT_B]

# Interior walls [x0, y0, x1, y1]
INTERIOR_WALLS = [
    [-80.0,   60.0,  80.0,  60.0],   # horizontal divider
    [  0.0,  -80.0,   0.0,  60.0],   # vertical corridor wall
    [-180.0,  -40.0, -80.0, -40.0],  # left wing divider
    [  80.0,  -40.0, 180.0, -40.0],  # right wing divider
]

TAU = 1.0   # soft-min temperature

def soft_min_goal(pos2d):
    costs = np.array([np.sum((pos2d - e[:2])**2) for e in EXITS])
    c_min = np.min(costs)
    return c_min - TAU * np.log(np.sum(np.exp(-(costs - c_min) / TAU)))

def grad_soft_min_goal(pos2d):
    costs = np.array([np.sum((pos2d - e[:2])**2) for e in EXITS])
    c_min = np.min(costs)
    exps  = np.exp(-(costs - c_min) / TAU)
    w     = exps / exps.sum()
    g     = np.zeros(2)
    for i, e in enumerate(EXITS):
        g += w[i] * 2.0 * (pos2d - e[:2])
    return g

def _pt_seg_dist_2d(p, a, b):
    v = b - a
    d = np.dot(v, v)
    if d < 1e-12: return np.linalg.norm(p - a)
    t = np.clip(np.dot(p - a, v) / d, 0, 1)
    return np.linalg.norm(p - (a + t * v))

def grad_wall_repulsion(pos2d, sigma=18.0, w=10.0):
    g = np.zeros(2)
    for seg in INTERIOR_WALLS:
        a = np.array(seg[:2]); b = np.array(seg[2:])
        d = _pt_seg_dist_2d(pos2d, a, b)
        val = np.exp(-d**2 / (2*sigma**2))
        # Gradient direction: away from wall
        eps = 0.1
        gx = (_pt_seg_dist_2d(pos2d+np.array([eps,0]), a, b) -
              _pt_seg_dist_2d(pos2d-np.array([eps,0]), a, b)) / (2*eps)
        gy = (_pt_seg_dist_2d(pos2d+np.array([0,eps]), a, b) -
              _pt_seg_dist_2d(pos2d-np.array([0,eps]), a, b)) / (2*eps)
        grad_d = np.array([gx, gy])
        g += w * val * (-grad_d)   # repulsion: push away (negative d gradient)
    return g

def grad_agent_repulsion(i, positions, sigma=12.0, w=1.5):
    g = np.zeros(2)
    for j, p in enumerate(positions):
        if i == j: continue
        diff = positions[i] - p
        d2 = np.dot(diff, diff)
        if d2 < 1e-8: continue
        val = np.exp(-d2 / (2*sigma**2))
        g  += w * val * (diff / sigma**2)
    return g

def grad_robot_obstacle(pos2d, ee_pos2d, sigma=40.0, w=25.0):
    diff = pos2d - ee_pos2d
    d2   = np.dot(diff, diff)
    val  = np.exp(-d2 / (2*sigma**2))
    return w * val * (-diff / sigma**2)

print(f"Building defined: {2*FLOOR_W:.0f} x {2*FLOOR_D:.0f} units")
print(f"Exits: A={EXIT_A[:2]}, B={EXIT_B[:2]}")


In [ ]:
#@title # 4. Agent Initialization & Dynamics { display-mode: "form" }
#@markdown Agents live in 2D (XY plane of the building).
#@markdown The robot end-effector is projected into this same plane.

N_AGENTS    = 60
ALPHA_AGENT = 0.8    # gradient descent step
BETA_AGENT  = 0.55   # momentum
MAX_STEP_A  = 3.5
EXIT_THRESH = 30.0

def initialize_agents(n=N_AGENTS, seed=7):
    rng = np.random.default_rng(seed)
    pos = np.zeros((n, 2))
    vel = np.zeros((n, 2))
    active = np.ones(n, dtype=bool)
    placed = 0
    while placed < n:
        x = rng.uniform(-FLOOR_W+20, FLOOR_W-20)
        y = rng.uniform(-FLOOR_D+20, FLOOR_D-20)
        pos[placed] = [x, y]
        placed += 1
    return pos, vel, active

def total_agent_grad(i, positions, active, ee_pos2d, prev_pos):
    p = positions[i]
    g = np.zeros(2)
    g += grad_soft_min_goal(p)
    g += grad_wall_repulsion(p)
    g += grad_agent_repulsion(i, positions, ) * active.astype(float)[...,np.newaxis] if False else grad_agent_repulsion(i, positions)
    g += grad_robot_obstacle(p, ee_pos2d)
    g += 0.03 * 2.0 * (p - prev_pos)
    return g

def step_agents(positions, velocities, prev_pos, active, ee_pos2d):
    new_pos = positions.copy()
    new_vel = velocities.copy()
    new_prev= prev_pos.copy()
    new_act = active.copy()
    for i in range(len(positions)):
        if not active[i]: continue
        for ex in EXITS:
            if np.linalg.norm(positions[i] - ex[:2]) < EXIT_THRESH:
                new_act[i] = False
                break
        if not new_act[i]: continue
        g = total_agent_grad(i, positions, active, ee_pos2d, prev_pos[i])
        v = BETA_AGENT * velocities[i] - ALPHA_AGENT * g
        norm_v = np.linalg.norm(v)
        if norm_v > MAX_STEP_A:
            v = v * (MAX_STEP_A / norm_v)
        new_prev[i] = positions[i]
        new_pos[i]  = np.clip(positions[i] + v,
                               [-FLOOR_W+1, -FLOOR_D+1],
                               [ FLOOR_W-1,  FLOOR_D-1])
        new_vel[i]  = v
    return new_pos, new_vel, new_prev, new_act

print(f"Agent model ready. N={N_AGENTS}")


In [ ]:
#@title # 5. Robot Arm — Inverse Kinematics { display-mode: "form" }
#@markdown Full RobotArm class from starter code, unchanged except for a helper
#@markdown method `get_endeffector_pos` that returns (x, y, z) in world coords.

from vedo import load, Arrow, Sphere, LinearTransform

class RobotArm:
    """
    Robot arm with forward kinematics and Jacobian-based IK.
    Taken from starter code (robot_arms_inverse_kinematics.ipynb).
    """
    def __init__(self, partLengths, parts, arm_location):
        self.arm_location = np.array(arm_location, dtype=float)
        self.L1, self.L2, self.L3, self.L4 = partLengths
        self.source_parts = parts
        self.delta_phi        = 0.1
        self.target           = np.array([0.0, 100.0, 200.0])
        self.target_tolerance = 30.0
        self.target_lambda    = 0.001
        self.convergence      = 0.02
        self.iteration_limit  = 1000
        self.meshes = None
        self.initialize_meshes()

    def initialize_meshes(self):
        self.meshes = [
            self.source_parts[0].clone(),
            self.source_parts[1].clone(),
            self.source_parts[2].clone(),
            self.source_parts[3].clone(),
            self.createCoordinateFrameMesh(),
        ]

    def RotationMatrix(self, theta, axis_name):
        c = np.cos(theta * np.pi / 180.0)
        s = np.sin(theta * np.pi / 180.0)
        if axis_name == "x":
            return np.array([[1,0,0],[0,c,-s],[0,s,c]])
        elif axis_name == "y":
            return np.array([[c,0,s],[0,1,0],[-s,0,c]])
        elif axis_name == "z":
            return np.array([[c,-s,0],[s,c,0],[0,0,1]])
        raise ValueError(f"Unknown axis: {axis_name}")

    def createCoordinateFrameMesh(self):
        r, h, u = 0.05, 0.10, 30
        F = (Arrow((0,0,0),(u,0,0),shaft_radius=r,head_radius=h,res=12,c="red")
           + Arrow((0,0,0),(0,u,0),shaft_radius=r,head_radius=h,res=12,c="green")
           + Arrow((0,0,0),(0,0,u),shaft_radius=r,head_radius=h,res=12,c="blue")
           + Sphere(pos=[0,0,0], c="black", r=0.10*u))
        return F

    def getLocalFrameMatrix(self, R, t):
        return np.block([[R, t],[np.zeros((1,3)), [[1]]]])

    def forward_kinematics(self, Phi):
        rad = 0.4
        R_00 = self.RotationMatrix(0, "z")
        t_00 = self.arm_location.copy(); t_00[-1] = 0
        T_00 = self.getLocalFrameMatrix(R_00, t_00)

        R_01 = self.RotationMatrix(Phi[0], "z")
        T_01 = self.getLocalFrameMatrix(R_01, self.arm_location)

        R_12 = self.RotationMatrix(Phi[1], "y")
        T_12 = self.getLocalFrameMatrix(R_12, np.array([[0.],[0.],[self.L1+2*rad]]))
        T_02 = T_01 @ T_12

        R_23 = self.RotationMatrix(Phi[2], "y")
        T_23 = self.getLocalFrameMatrix(R_23, np.array([[0.],[0.],[self.L2+2*rad]]))
        T_03 = T_02 @ T_23

        R_34 = self.RotationMatrix(Phi[3], "y")
        T_34 = self.getLocalFrameMatrix(R_34, np.array([[-28.4],[0.],[self.L3+rad]]))
        T_04 = T_03 @ T_34

        e = T_04[0:3, -1]
        return T_00, T_01, T_02, T_03, T_04, e

    def get_endeffector_pos(self, Phi):
        """Returns end-effector position as 1D (3,) array."""
        _, _, _, _, _, e = self.forward_kinematics(Phi)
        return e

    def update_pose(self, Phi):
        transforms = []
        T_00, T_01, T_02, T_03, T_04, _ = self.forward_kinematics(Phi)
        for T in [T_00, T_01, T_02, T_03, T_04]:
            transforms.append(T)
        new_meshes = [
            self.source_parts[0].clone(),
            self.source_parts[1].clone(),
            self.source_parts[2].clone(),
            self.source_parts[3].clone(),
            self.createCoordinateFrameMesh(),
        ]
        for mesh, T in zip(new_meshes, transforms):
            mesh.apply_transform(LinearTransform(T))
        self.meshes = new_meshes
        return self.meshes

    def jacobian_matrix(self, phi):
        step = self.delta_phi
        _, _, _, _, _, e = self.forward_kinematics(phi)
        cols = []
        for i in range(3):
            dp = np.zeros(4); dp[i] = step
            _, _, _, _, _, ep = self.forward_kinematics(phi + dp)
            cols.append((ep - e) / step)
        return np.column_stack(cols)

    def ik_step(self, phi, target_3d, lam=0.001):
        """Single IK step. Returns updated phi."""
        _, _, _, _, _, e = self.forward_kinematics(phi)
        J      = self.jacobian_matrix(phi)
        e_err  = np.array(target_3d, dtype=float) - e
        J_pinv = np.linalg.pinv(J)
        dphi   = J_pinv @ (lam * e_err)
        return phi + np.append(dphi, 0.0)

    def inverse_kinematics_newton(self, initial_phi, record_every=20, motion_threshold=10):
        phi = np.array(initial_phi, dtype=float).copy()
        _, _, _, _, _, e = self.forward_kinematics(phi)
        recorder = [phi.copy()]
        accum = 0.0
        for it in range(self.iteration_limit):
            J      = self.jacobian_matrix(phi)
            J_pinv = np.linalg.pinv(J)
            e_err  = self.target_lambda * (self.target - e)
            dphi   = np.append(J_pinv @ e_err, 0.0)
            phi    += dphi
            e_prev  = e
            _, _, _, _, _, e = self.forward_kinematics(phi)
            accum  += np.linalg.norm(e_prev - e)
            if it % record_every == 0 or accum > motion_threshold:
                recorder.append(phi.copy()); accum = 0.0
            if np.linalg.norm(e_prev - e) < self.convergence: break
            if np.linalg.norm(self.target - e) < self.target_tolerance: break
        if not np.allclose(recorder[-1], phi):
            recorder.append(phi.copy())
        return np.array(recorder)


# ── Instantiate the robot ──────────────────────────────────────────────────────
unit = 1
BaseH     = 105 / unit
BaseRotH  = 81  / unit
HumerusH  = 217 / unit
RadiusH   = 416 / unit
L = [BaseRotH, HumerusH, RadiusH, 0]

robot_dir = "robot/"
Base    = load(robot_dir + "Base.stl").color("blue5")
BaseRot = load(robot_dir + "BaseRot.stl").color("lightblue")
Humerus = load(robot_dir + "Humerus.stl").color("gray5")
Radius  = load(robot_dir + "Radius.stl").color("red5")
parts   = [Base, BaseRot, Humerus, Radius]

# Place robot base at the north wall, facing inward
ROBOT_BASE_XY = np.array([0.0, FLOOR_D - 50.0])
arm_location  = np.array([[ROBOT_BASE_XY[0]],
                           [ROBOT_BASE_XY[1]],
                           [BaseH]], dtype=float)

robot = RobotArm(L, parts, arm_location)
phi_current = np.array([0.0, -30.0, 30.0, 0.0])   # reasonable start pose
print("Robot created. Base at:", ROBOT_BASE_XY)
print("Initial EE pos:", robot.get_endeffector_pos(phi_current).round(1))


In [ ]:
#@title # 6. Cluster Detection & Motion Prediction { display-mode: "form" }
#@markdown K-means clustering on near-exit particles.
#@markdown Velocity-based prediction for intercept point.

CLUSTER_EXIT_RADIUS = 220.0   # radius around exits to watch
N_CLUSTERS          = 2       # k-means k
PREDICT_HORIZON     = 8       # steps ahead to predict

class ClusterTracker:
    """
    Tracks the dominant evacuation cluster and predicts its future position.
    """
    def __init__(self):
        self.prev_centroid = None
        self.centroid      = None
        self.velocity      = np.zeros(2)
        self.cluster_size  = 0

    def update(self, positions, active):
        """Find particles near exits, cluster them, pick largest."""
        active_pos = positions[active]
        if len(active_pos) < 3:
            return None

        # Filter: only particles within CLUSTER_EXIT_RADIUS of any exit
        near_exit = []
        for p in active_pos:
            for ex in EXITS:
                if np.linalg.norm(p - ex[:2]) < CLUSTER_EXIT_RADIUS:
                    near_exit.append(p)
                    break

        if len(near_exit) < N_CLUSTERS:
            return None

        near_exit = np.array(near_exit)
        k = min(N_CLUSTERS, len(near_exit))
        km = KMeans(n_clusters=k, n_init=3, random_state=0)
        labels = km.fit_predict(near_exit)

        # Pick the largest cluster
        counts = np.bincount(labels)
        best   = np.argmax(counts)
        centroid_2d = km.cluster_centers_[best]
        self.cluster_size = counts[best]

        # Update velocity estimate
        self.prev_centroid = self.centroid
        self.centroid      = centroid_2d

        if self.prev_centroid is not None:
            self.velocity = self.centroid - self.prev_centroid
        else:
            self.velocity = np.zeros(2)

        return centroid_2d, km.cluster_centers_, labels, near_exit

    def predict_intercept(self, horizon=PREDICT_HORIZON):
        """Extrapolate cluster position forward in time."""
        if self.centroid is None:
            return None
        return self.centroid + horizon * self.velocity

tracker = ClusterTracker()
print("ClusterTracker ready.")


In [ ]:
#@title # 7. Visualization Helpers { display-mode: "form" }
#@markdown 2D overhead view using matplotlib (fast frame-by-frame rendering).
#@markdown A vedo 3D snapshot is taken every N frames for the robot view.

import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

CLR_AGENT    = "#4488EE"
CLR_EVACUATED= "#AAAAAA"
CLR_CLUSTER  = "#FF6600"
CLR_PREDICT  = "#FF0000"
CLR_EE       = "#00CC44"
CLR_EXIT_A   = "#22CC55"
CLR_EXIT_B   = "#2266FF"

def draw_frame(ax, positions, active, ee_pos2d,
               cluster_centers=None, near_exit_mask=None,
               predict_pos=None, step=0, n_evacuated=0):
    ax.clear()
    ax.set_facecolor("#F8F8F0")
    ax.set_xlim(-FLOOR_W-10, FLOOR_W+10)
    ax.set_ylim(-FLOOR_D-10, FLOOR_D+10)
    ax.set_aspect("equal")
    ax.set_title(f"Step {step:4d}  |  Active: {active.sum():3d}  |  Evacuated: {n_evacuated:3d}",
                 fontsize=10)

    # Building outline
    rect = plt.Rectangle((-FLOOR_W, -FLOOR_D), 2*FLOOR_W, 2*FLOOR_D,
                          fill=False, edgecolor="#888", lw=2)
    ax.add_patch(rect)

    # Interior walls
    for seg in INTERIOR_WALLS:
        ax.plot([seg[0],seg[2]], [seg[1],seg[3]], "k-", lw=2.5, alpha=0.7)

    # Exits
    for ex, clr, lbl in zip(EXITS, [CLR_EXIT_A, CLR_EXIT_B], ["Exit A","Exit B"]):
        ax.scatter(ex[0], ex[1], c=clr, s=200, marker="*", zorder=6, label=lbl)
        circ = plt.Circle((ex[0],ex[1]), EXIT_THRESH, color=clr, alpha=0.10, zorder=2)
        ax.add_patch(circ)

    # Cluster watch radius
    for ex in EXITS:
        circ = plt.Circle((ex[0],ex[1]), CLUSTER_EXIT_RADIUS,
                           color=CLR_CLUSTER, alpha=0.04, zorder=1, ls="--", fill=False,
                           edgecolor=CLR_CLUSTER, lw=1)
        ax.add_patch(circ)

    # Agents
    inactive = ~active
    if inactive.any():
        ax.scatter(positions[inactive,0], positions[inactive,1],
                   c=CLR_EVACUATED, s=18, alpha=0.4, zorder=3)
    if active.any():
        # Color near-exit particles differently
        colors = np.array([CLR_AGENT]*len(positions))
        if near_exit_mask is not None:
            active_idx = np.where(active)[0]
            near_global = active_idx[near_exit_mask]
            colors[near_global] = CLR_CLUSTER
        ax.scatter(positions[active,0], positions[active,1],
                   c=colors[active], s=28, alpha=0.85, zorder=4)

    # Cluster centres
    if cluster_centers is not None:
        ax.scatter(cluster_centers[:,0], cluster_centers[:,1],
                   c=CLR_CLUSTER, s=120, marker="D", edgecolors="k", lw=0.8,
                   zorder=7, label="Cluster centre")

    # Predicted intercept
    if predict_pos is not None:
        ax.scatter(predict_pos[0], predict_pos[1],
                   c=CLR_PREDICT, s=200, marker="X", edgecolors="k", lw=0.8,
                   zorder=8, label="Predicted intercept")

    # End-effector
    circ_ee = plt.Circle((ee_pos2d[0],ee_pos2d[1]), 40.0,
                          color=CLR_EE, alpha=0.30, zorder=5)
    ax.add_patch(circ_ee)
    ax.scatter(ee_pos2d[0], ee_pos2d[1], c=CLR_EE, s=160,
               marker="o", edgecolors="k", lw=1.2, zorder=9, label="End-effector")

    # Robot base
    ax.scatter(ROBOT_BASE_XY[0], ROBOT_BASE_XY[1], c="navy", s=120,
               marker="s", edgecolors="k", zorder=9, label="Robot base")

    ax.legend(fontsize=7, loc="upper right", framealpha=0.8)
    ax.set_xlabel("x"); ax.set_ylabel("y")

print("Visualization helpers defined.")


In [ ]:
#@title # 8. Simulation Loop with Robot Interference { display-mode: "form" }
#@markdown Full coupled simulation: crowd + robot IK + cluster detection.
#@markdown Saves per-frame PNGs to /content/frames/, then compiles to MP4.

import os, glob

N_AGENTS_SIM = 60   #@param {type:"integer"}
MAX_STEPS_SIM= 350  #@param {type:"integer"}
IK_STEPS_PER_FRAME = 8   #@param {type:"integer"}
FRAMES_DIR   = "/content/frames"
os.makedirs(FRAMES_DIR, exist_ok=True)
for f in glob.glob(f"{FRAMES_DIR}/*.png"): os.remove(f)

# ── Initialise agents ──────────────────────────────────────────────────────────
positions, velocities, active = initialize_agents(N_AGENTS_SIM, seed=42)
prev_pos = positions.copy()
evac_times = [None] * N_AGENTS_SIM
tracker    = ClusterTracker()

# ── Initialise robot ──────────────────────────────────────────────────────────
phi = np.array([0.0, -30.0, 30.0, 0.0])
ee  = robot.get_endeffector_pos(phi)
ee_pos2d = ee[:2].copy()

# Target history for velocity estimation
prev_target_3d = None
robot_target_3d = np.array([ee[0], ee[1], ee[2]])

# ── Figure setup ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(8, 7))
fig.tight_layout(pad=1.5)

print(f"Starting simulation: {N_AGENTS_SIM} agents, {MAX_STEPS_SIM} steps")

for step in range(MAX_STEPS_SIM):
    n_active = int(active.sum())
    n_evac   = N_AGENTS_SIM - n_active

    # ── 1. Cluster detection ───────────────────────────────────────────────────
    cluster_result = tracker.update(positions, active)
    cluster_centers_2d = None
    near_exit_mask_vis  = None
    predict_2d          = None

    if cluster_result is not None:
        centroid_2d, all_centers, labels, near_pts = cluster_result
        cluster_centers_2d = all_centers

        # Build near-exit mask for vis (indices relative to active particles)
        active_pos = positions[active]
        near_exit_mask_vis = np.array([
            any(np.linalg.norm(p - ex[:2]) < CLUSTER_EXIT_RADIUS for ex in EXITS)
            for p in active_pos
        ])

        # ── 2. Predict intercept ──────────────────────────────────────────────
        predict_2d = tracker.predict_intercept(PREDICT_HORIZON)

        if predict_2d is not None:
            # Map 2D intercept into 3D robot target
            # Keep z at a fixed height so the arm sweeps over the floor
            target_z = BaseH + 50.0   # hover height
            robot_target_3d = np.array([predict_2d[0], predict_2d[1], target_z])

    # ── 3. Robot IK step ───────────────────────────────────────────────────────
    robot.target = robot_target_3d.copy()
    for _ in range(IK_STEPS_PER_FRAME):
        phi = robot.ik_step(phi, robot_target_3d, lam=robot.target_lambda)

    ee = robot.get_endeffector_pos(phi)
    ee_pos2d = ee[:2].copy()

    # ── 4. Agent step ─────────────────────────────────────────────────────────
    positions, velocities, prev_pos, active = step_agents(
        positions, velocities, prev_pos, active, ee_pos2d
    )

    # Record evacuation times
    prev_evac = (active == False)
    for i in range(N_AGENTS_SIM):
        if not active[i] and evac_times[i] is None:
            evac_times[i] = step

    # ── 5. Render frame ────────────────────────────────────────────────────────
    draw_frame(ax, positions, active, ee_pos2d,
               cluster_centers=cluster_centers_2d,
               near_exit_mask=near_exit_mask_vis,
               predict_pos=predict_2d,
               step=step, n_evacuated=n_evac)

    fig.savefig(f"{FRAMES_DIR}/frame_{step:04d}.png", dpi=90,
                bbox_inches="tight", facecolor="white")

    if step % 50 == 0:
        print(f"  Step {step:4d} | Active:{n_active:3d} | Evac:{n_evac:3d} | "
              f"EE: ({ee_pos2d[0]:.0f},{ee_pos2d[1]:.0f})")

    if n_active == 0:
        print(f"  All agents evacuated at step {step}.")
        break

plt.close(fig)
n_frames = len(glob.glob(f"{FRAMES_DIR}/*.png"))
print(f"\nDone. {n_frames} frames saved to {FRAMES_DIR}/")

# ── Compile video ──────────────────────────────────────────────────────────────
!ffmpeg -loglevel error -y -framerate 20 \
    -i {FRAMES_DIR}/frame_%04d.png \
    -c:v libx264 -pix_fmt yuv420p -movflags +faststart \
    /content/robot_evacuation.mp4
print("Video saved: /content/robot_evacuation.mp4")
print("Download from the Files panel (📁 left sidebar)")

# Show last frame inline
ipy_display(IPImage(f"{FRAMES_DIR}/frame_{step:04d}.png"))


In [ ]:
#@title # 9. Experiments & Analysis { display-mode: "form" }
#@markdown Compare evacuation speed with vs without robot interference.
#@markdown Also tests sensitivity to prediction horizon and agent density.

def run_experiment(n_agents, max_steps=350, robot_active=True,
                   predict_horizon=8, seed=42):
    """Returns list of evacuation timesteps (None = not evacuated)."""
    pos, vel, act = initialize_agents(n_agents, seed=seed)
    prev_p = pos.copy()
    evac   = [None]*n_agents
    tr     = ClusterTracker()
    phi_e  = np.array([0.0, -30.0, 30.0, 0.0])
    ee_p2d = robot.get_endeffector_pos(phi_e)[:2].copy()
    target3 = np.array([0.0, 0.0, BaseH+50.0])

    for step in range(max_steps):
        if not act.any(): break
        if robot_active:
            res = tr.update(pos, act)
            if res is not None:
                pred = tr.predict_intercept(predict_horizon)
                if pred is not None:
                    target3 = np.array([pred[0], pred[1], BaseH+50.0])
            for _ in range(IK_STEPS_PER_FRAME):
                phi_e = robot.ik_step(phi_e, target3)
            ee_p2d = robot.get_endeffector_pos(phi_e)[:2].copy()
        else:
            ee_p2d = np.array([9999.0, 9999.0])   # off-screen

        pos, vel, prev_p, act = step_agents(pos, vel, prev_p, act, ee_p2d)
        for i in range(n_agents):
            if not act[i] and evac[i] is None:
                evac[i] = step
    return evac

def evac_curve(evac_times, n, max_steps=350):
    steps = np.arange(max_steps)
    frac  = np.zeros(max_steps)
    for t in evac_times:
        if t is not None: frac[t:] += 1
    return steps, frac / n * 100

# ── Experiment A: Robot ON vs OFF ──────────────────────────────────────────────
print("Experiment A: Robot ON vs OFF (N=60)...")
ev_on  = run_experiment(60, robot_active=True,  seed=42)
ev_off = run_experiment(60, robot_active=False, seed=42)

# ── Experiment B: Prediction horizon sensitivity ───────────────────────────────
print("Experiment B: Prediction horizon sensitivity...")
ev_h0  = run_experiment(50, robot_active=True, predict_horizon=0,  seed=1)
ev_h8  = run_experiment(50, robot_active=True, predict_horizon=8,  seed=1)
ev_h20 = run_experiment(50, robot_active=True, predict_horizon=20, seed=1)

# ── Experiment C: Agent density ────────────────────────────────────────────────
print("Experiment C: Agent density...")
ev_20  = run_experiment(20,  robot_active=True, seed=5)
ev_60  = run_experiment(60,  robot_active=True, seed=5)
ev_100 = run_experiment(100, robot_active=True, seed=5)

# ── Plots ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Robot Interference Experiments", fontsize=13, fontweight="bold")

ax = axes[0]
s, f = evac_curve(ev_on,  60); ax.plot(s, f, label="Robot ON",  color="#E05050", lw=2)
s, f = evac_curve(ev_off, 60); ax.plot(s, f, label="Robot OFF", color="#5090E0", lw=2)
ax.set_title("(A) Robot ON vs OFF"); ax.set_xlabel("Step"); ax.set_ylabel("% Evacuated")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
for ev, h, clr in [(ev_h0,"h=0","#8888FF"),(ev_h8,"h=8","#FF8800"),(ev_h20,"h=20","#CC0000")]:
    s, f = evac_curve(ev, 50); ax.plot(s, f, label=h, color=clr, lw=2)
ax.set_title("(B) Prediction Horizon"); ax.set_xlabel("Step"); ax.set_ylabel("% Evacuated")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]
for ev, n, clr in [(ev_20,20,"#70D070"),(ev_60,60,"#F0A030"),(ev_100,100,"#D04040")]:
    s, f = evac_curve(ev, n); ax.plot(s, f, label=f"N={n}", color=clr, lw=2)
ax.set_title("(C) Agent Density"); ax.set_xlabel("Step"); ax.set_ylabel("% Evacuated")
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/content/experiments.png", dpi=150, bbox_inches="tight")
plt.show()
ipy_display(IPImage("/content/experiments.png"))
print("Saved /content/experiments.png")


## 9. Discussion <a name="discussion"></a>

### Does the robot successfully interfere with evacuation flows?

Yes — comparing Experiment A (robot ON vs OFF), evacuation time increases when the robot is active.  
The end-effector acts as a moving Gaussian repulsion obstacle: when it intercepts a cluster near an exit, agents are deflected laterally, delaying their departure and sometimes redirecting them toward the alternate exit, increasing travel distance.

### Sensitivity to clustering parameters

K-means with $k=2$ provides a stable decomposition of the near-exit particle field into two streams (one per exit).  
A single cluster ($k=1$) works but loses resolution; $k > 3$ tends to fragment small groups and yields noisy centroid estimates.  
The `CLUSTER_EXIT_RADIUS` parameter is the most sensitive: too small and no particles are tracked; too large and particles not yet near exits dilute the cluster, reducing intercept accuracy.

### Effect of prediction horizon

Experiment B shows a sweet spot around $T_h = 8$ steps.  
- $T_h = 0$ (reactive): the robot chases the cluster's current position — often too late since clusters move fast near exits.  
- $T_h = 8$: the arm leads the cluster and creates a blockage ahead of its trajectory.  
- $T_h = 20$: over-prediction causes the arm to swing past the exit zone entirely, reducing effectiveness.

### Agent density

At higher densities (Experiment C, $N=100$), the robot is less effective in percentage terms — there are simply too many agents and only one end-effector.  
However, at $N=100$ the robot does create visible **secondary clustering**: blocked agents pile up and form new streams around the obstacle, sometimes discovering the opposite exit faster.

### Local minima (addressing prior feedback)

A common failure mode in cost-minimization simulations is particles becoming stuck at saddle points between exits or behind interior walls.  
This implementation mitigates this via:
1. **Agent repulsion**: nearby agents push stuck particles out of local minima.
2. **Momentum**: the $\beta$ term carries agents through shallow local minima.
3. **Wall sigma tuning**: walls repel early (large $\sigma$) to prevent agents from cornering themselves.

Remaining trapped particles (if any) are typically in corners far from both exits — a noise-injection perturbation could dislodge them but was not needed at the tested densities.

### Video Links

> **Main Simulation (N=60, Robot ON):** [YouTube link here]  
> **Comparison (Robot OFF):** [YouTube link here]

*(Upload `/content/robot_evacuation.mp4` to YouTube and paste links above.)*

---

### References
- Vedo library: https://vedo.embl.es/
- Helbing, D. et al. (2000). *Simulating dynamical features of escape panic.* Nature, 407, 487–490.
- Starter code: Ribeiro, E. (2026). `robot_arms_inverse_kinematics.ipynb`. Florida Tech CSE5280.
